# Operational Schema Setup — NorthPeak Retail
Creates writable Postgres tables in `northpeak_app` schema,
distinct from the read-only synced tables in `northpeak` schema.

In [0]:
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()

PROJECT = 'northpeak'
BRANCH = 'production'
HOST = 'ep-little-union-d20vlyjv.database.us-east-1.cloud.databricks.com'
DATABASE = 'databricks_postgres'
print(f'Lakebase project: {PROJECT}')
print(f'Host: {HOST}')

In [0]:
# Create writable schema for application state
# (read-only synced tables live in 'northpeak' schema)
DDL = '''
CREATE SCHEMA IF NOT EXISTS northpeak_app;

CREATE TABLE IF NOT EXISTS northpeak_app.transfer_actions (
    action_id UUID DEFAULT gen_random_uuid() PRIMARY KEY,
    source_store_id TEXT NOT NULL,
    destination_store_id TEXT NOT NULL,
    product_id TEXT NOT NULL,
    units_requested INTEGER NOT NULL,
    units_approved INTEGER,
    status TEXT NOT NULL DEFAULT 'proposed'
        CHECK (status IN ('proposed','approved','rejected','committed','cancelled')),
    requested_by TEXT,
    approved_by TEXT,
    reason TEXT,
    created_at TIMESTAMPTZ NOT NULL DEFAULT now(),
    updated_at TIMESTAMPTZ NOT NULL DEFAULT now(),
    approved_at TIMESTAMPTZ,
    committed_at TIMESTAMPTZ
);

CREATE TABLE IF NOT EXISTS northpeak_app.inventory_alerts (
    alert_id UUID DEFAULT gen_random_uuid() PRIMARY KEY,
    store_id TEXT NOT NULL,
    product_id TEXT NOT NULL,
    alert_type TEXT NOT NULL,
    severity TEXT NOT NULL DEFAULT 'medium',
    message TEXT,
    acknowledged BOOLEAN DEFAULT FALSE,
    acknowledged_by TEXT,
    created_at TIMESTAMPTZ NOT NULL DEFAULT now(),
    acknowledged_at TIMESTAMPTZ
);

CREATE TABLE IF NOT EXISTS northpeak_app.product_substitutes (
    substitute_id UUID DEFAULT gen_random_uuid() PRIMARY KEY,
    product_id TEXT NOT NULL,
    substitute_product_id TEXT NOT NULL,
    substitution_type TEXT NOT NULL
        CHECK (substitution_type IN ('equivalent','upgrade','downgrade','similar')),
    confidence_score NUMERIC(3,2) CHECK (confidence_score BETWEEN 0 AND 1),
    reason TEXT,
    is_active BOOLEAN DEFAULT TRUE,
    created_at TIMESTAMPTZ NOT NULL DEFAULT now(),
    UNIQUE(product_id, substitute_product_id)
);

-- Enable REPLICA IDENTITY for CDF reverse sync
ALTER TABLE northpeak_app.transfer_actions REPLICA IDENTITY FULL;
ALTER TABLE northpeak_app.inventory_alerts REPLICA IDENTITY FULL;
'''
print(DDL)

In [0]:
# Execute DDL via SDK
import psycopg2
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()

ENDPOINT = f'projects/{PROJECT}/branches/{BRANCH}/endpoints/primary'
token = w.postgres.generate_database_credential(endpoint=ENDPOINT).token
username = w.current_user.me().user_name
print(f'OAuth credential generated for: {ENDPOINT}')
print(f'Connecting as: {username}')

conn = psycopg2.connect(
    host=HOST, dbname=DATABASE, user=username,
    password=token, sslmode='require'
)
conn.autocommit = True
cur = conn.cursor()
for stmt in DDL.strip().split(';'):
    stmt = stmt.strip()
    if stmt:
        cur.execute(stmt)
        print(f'OK: {stmt[:60]}...')
cur.close()
conn.close()
print('\nAll DDL executed successfully.')

In [0]:
# Verify: list tables in both schemas
import psycopg2
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()

ENDPOINT = f'projects/{PROJECT}/branches/{BRANCH}/endpoints/primary'
token = w.postgres.generate_database_credential(endpoint=ENDPOINT).token
username = w.current_user.me().user_name

conn = psycopg2.connect(host=HOST, dbname=DATABASE, user=username,
                        password=token, sslmode='require')
cur = conn.cursor()
cur.execute("""SELECT schemaname, tablename FROM pg_tables
               WHERE schemaname IN ('northpeak','northpeak_app')
               ORDER BY schemaname, tablename""")
rows = cur.fetchall()
print(f"{'Schema':<15} {'Table':<25} {'Type'}")
print('-' * 55)
for schema, table in rows:
    ttype = 'SYNCED (read-only)' if schema == 'northpeak' else 'WRITABLE (app state)'
    print(f'{schema:<15} {table:<25} {ttype}')
cur.close()
conn.close()
print('\nExecution proof: writable tables coexist with read-only synced tables.')